# 13_v5c_3_3a_more_hedge — V5C 3.3a 进一步增加对冲

> 测试: 从 QQQ 15% 拿出 5% 给 DBMF, DBMF 增至 10%

## 配置对比

| 标的 | V5C 3.3 | V5C 3.3a |
|---|---|---|
| VOO | 10% | 10% |
| QQQ | 15% | **10%** |
| HQH | 10% | 10% |
| XLV | 10% | 10% |
| GLDM | 20% | 20% |
| BCX | 10% | 10% |
| **DBMF** | **5%** | **10%** |
| VGSH | 20% | 20% |
| **结构** | **45/35/20** | **40/40/20** |

## 核心问题

1. 进攻减 5pp + 对冲加 5pp → Sharpe 是否仍可接受？
2. DBMF 从 5% 加到 10% → 2022 滞胀更好但 7Y 整体 CAGR 损失多少？
3. 这是 V5C 3.3 的更稳健版还是 over-hedged？

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickers = ['VFINX','QQQ','HQH','XLV','VFITX','GLD','GC=F','DBC','PCRIX','DBMF','AQRIX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True)['Close']

def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'], 'QQQ': raw['QQQ'],
    'HQH': raw['HQH'], 'XLV': raw['XLV'],
    'VGSH': raw['VFITX'],
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX': synthesize(raw['DBC'], raw['PCRIX']),
    'DBMF': raw['DBMF'],
    'AQRIX': raw['AQRIX'],
})

In [ ]:
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub = returns_df[used].dropna()
    target = np.array([target_weights[t] for t in used]); target = target/target.sum()
    cw = target.copy(); pr=[]; rd=[sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw-target))*100 >= threshold_pp:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, dates, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    downside = rs[rs<0]
    sortino = (cagr-0.04)/(downside.std()*np.sqrt(252))
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,'Sortino':sortino,
            'Max DD':dd.min(),'Calmar':cagr/abs(dd.min()),'Rebalances':len(dates)-1}

def show_compare(metric_list):
    cols = ['CAGR','Vol','Sharpe','Sortino','Max DD','Calmar']
    print(f"{'Metric':<10}", end='')
    for m in metric_list: print(f"  {m['Name']:<22}", end='')
    print()
    print('-'*100)
    for col in cols:
        fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
        line = f'{col:<10}'
        for m in metric_list:
            line += f'  {fmt.format(m[col]):<22}'
        print(line)
    print(f"{'Rebalances':<10}", end='')
    for m in metric_list: print(f"  {str(m['Rebalances']):<22}", end='')
    print()

In [ ]:
# ============================================================
# 测试 1 (核心): 7Y - 实际 DBMF 数据
# ============================================================
print('=' * 100)
print('测试 1 (核心): 7Y - V5C 3.1 vs V5C 3.3 vs V5C 3.3a')
print('=' * 100)

data_7y = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX','DBMF']].dropna()
ret_7y = data_7y.pct_change().dropna()
print(f'窗口: {data_7y.index[0].date()} → {data_7y.index[-1].date()} ({len(data_7y)/252:.1f} 年)')

V5C_3_1  = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'VGSH':0.20}
V5C_3_3  = {'VOO':0.10,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'DBMF':0.05,'VGSH':0.20}
V5C_3_3a = {'VOO':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

for name, w in [('V5C 3.1', V5C_3_1), ('V5C 3.3', V5C_3_3), ('V5C 3.3a', V5C_3_3a)]:
    print(f'  {name} 权重和: {sum(w.values()):.2f}')

r_31, d_31 = simulate_rebalance(ret_7y, V5C_3_1)
r_33, d_33 = simulate_rebalance(ret_7y, V5C_3_3)
r_33a, d_33a = simulate_rebalance(ret_7y, V5C_3_3a)

show_compare([
    metrics(r_31, d_31, '3.1 (45/30/20)'),
    metrics(r_33, d_33, '3.3 (45/35/20)'),
    metrics(r_33a, d_33a, '3.3a (40/40/20)'),
])

print('\n关键时期表现 (7Y 窗口):')
events_7y = {
    '2020 COVID':         ('2020-02-19', '2020-04-30'),
    '2020 流动性极端':     ('2020-03-09', '2020-03-23'),
    '2020-2021 反弹':      ('2020-04-30', '2021-12-31'),
    '2022 Bear (全年)':    ('2022-01-01', '2022-12-31'),
    '2022 滞胀核心':        ('2022-01-01', '2022-09-30'),
    '2023-2024 AI 牛':    ('2023-01-01', '2024-12-31'),
    '2025 Q1 关税':         ('2025-01-01', '2025-04-30'),
}
print(f"{'时期':<22} {'3.1':>10} {'3.3':>10} {'3.3a':>10} {'3.3a-3.3':>11}")
for n, (s, e) in events_7y.items():
    a = (1 + r_31.loc[s:e]).prod() - 1
    b = (1 + r_33.loc[s:e]).prod() - 1
    c = (1 + r_33a.loc[s:e]).prod() - 1
    print(f'{n:<22} {a:>+9.2%}  {b:>+9.2%}  {c:>+9.2%}    {c-b:>+9.2%}')

In [ ]:
# ============================================================
# 测试 2: 16Y - 用 AQRIX 代理 (有效性存疑)
# ============================================================
print('=' * 100)
print('测试 2: 16Y - AQRIX 代理 (DBMF 长史代理)')
print('=' * 100)

data_16y = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX','AQRIX']].dropna()
ret_16y = data_16y.pct_change().dropna()
print(f'窗口: {data_16y.index[0].date()} → {data_16y.index[-1].date()} ({len(data_16y)/252:.1f} 年)')

V5C_3_3_proxy  = {'VOO':0.10,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'AQRIX':0.05,'VGSH':0.20}
V5C_3_3a_proxy = {'VOO':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'AQRIX':0.10,'VGSH':0.20}

r_31_16, d_31_16 = simulate_rebalance(ret_16y, V5C_3_1)
r_33_16, d_33_16 = simulate_rebalance(ret_16y, V5C_3_3_proxy)
r_33a_16, d_33a_16 = simulate_rebalance(ret_16y, V5C_3_3a_proxy)

show_compare([
    metrics(r_31_16, d_31_16, '3.1 (基准)'),
    metrics(r_33_16, d_33_16, '3.3-proxy (5% AQRIX)'),
    metrics(r_33a_16, d_33a_16, '3.3a-proxy (10% AQRIX)'),
])

In [ ]:
# ============================================================
# 净值 + 回撤 (7Y 窗口, 三组)
# ============================================================
cum_31  = (1 + r_31).cumprod()
cum_33  = (1 + r_33).cumprod()
cum_33a = (1 + r_33a).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(cum_31,  label='V5C 3.1 (基准 45/30/20)', linewidth=2, alpha=0.85)
axes[0].plot(cum_33,  label='V5C 3.3 (DBMF 5%, 45/35/20)', linewidth=2, alpha=0.85)
axes[0].plot(cum_33a, label='V5C 3.3a (DBMF 10%, 40/40/20)', linewidth=2, alpha=0.85)
axes[0].set_title('7Y 净值对比 (log scale)', fontsize=14)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for cum, label, color in [(cum_31, '3.1', 'C0'), (cum_33, '3.3', 'C2'), (cum_33a, '3.3a', 'C3')]:
    rm = cum.expanding().max()
    dd = (cum / rm) - 1
    axes[1].fill_between(dd.index, dd.values, 0, alpha=0.3, color=color, label=label)
axes[1].set_title('回撤对比', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 决策标准

**采纳 V5C 3.3a (40/40/20)** 仅当：
1. Sharpe 不显著低于 V5C 3.3（差距 < 0.03）
2. 2022 Bear 表现进一步改善 ≥ 1.5pp vs 3.3
3. Max DD 进一步改善 ≥ 0.5pp vs 3.3

**保持 V5C 3.3** 如果：
- DBMF 从 5% 加到 10% 反而拖累 Sharpe（DBMF 自身 7Y Sharpe 仅 0.43）
- 改善边际效用递减（即对冲层 35% → 40% 的边际价值已经很小）

## 预测

基于 DBMF 单标的 Sharpe 0.43 < 组合 Sharpe 0.93 的事实：
- 加大 DBMF 权重几乎一定会**降低整体 Sharpe**
- 但 2022 表现应当**进一步改善**（DBMF 翻倍）
- 取舍是 "风险调整后回报 vs 危机期表现"